In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path("/content/lora-activation-steering")
if not repo.exists():
    subprocess.run([
        "git", "clone", "--branch", "work",
        "https://github.com/danilov-e/lora-activation-steering.git", str(repo)
    ], check=True)
%cd /content/lora-activation-steering
%pip install -e ".[test]"


In [ ]:
subprocess.run([
    sys.executable, "-m", "pytest", "tests/test_core.py", "tests/test_data.py", "-q"
], check=True)


In [ ]:
from pprint import pprint
from src.data import load_persona_data, build_extraction_jobs, build_eval_jobs

data = load_persona_data("configs/persona_sycophantic.yaml")
jobs = build_extraction_jobs(data)
dev_jobs = build_eval_jobs(data, split="dev")

print("Extraction:", len(jobs), "Dev:", len(dev_jobs))
print("Source commit:", data["config"]["source"]["revision"])
print("Dev IDs:", data["config"]["split"]["dev"])
pprint(jobs[0])
pprint(jobs[1])

assert jobs[0]["pair_id"] == jobs[1]["pair_id"]
assert all(job["system"] is None for job in dev_jobs)


In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.prepare_jobs",
    "--split", "extract", "--output", "outputs/sycophantic_extract"
], check=True)


In [ ]:
from src.config import load_config
from src.model import load_model_and_tokenizer
from src.generation import generate_batch

config = load_config("configs/qwen25_15b.yaml")
model, tokenizer = load_model_and_tokenizer(config["model"])
print(type(model).__name__, model.device, model.dtype)


In [ ]:
smoke_settings = dict(config["generation"])
smoke_settings["max_new_tokens"] = 128

rows = generate_batch(
    model, tokenizer, jobs[:2], smoke_settings, seed=config["run"]["seed"]
)

for row in rows:
    print(row["id"], row["pair_id"], row["answer"], sep="\n")
    print("Tokens:", row["generated_tokens"], "Finish:", row["finish_reason"])
    print()
